In [1]:
import gc, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA, INTERIM_DATA
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.lstm import LSTMModel
from src.fully_connected_colab import read_parquet_safe, load_split_bundle
from src.model3_utils import (
    FEATURE_SETS, TARGET, LOOKBACK,
    assign_sequences_to_splits, scale_sequences,
    SequenceDataset, detect_device, compute_batch_size,
    train_seq_model, predict_seq,
    hw_predict_aligned, save_seq_run,
    print_config, print_feature_set_summary,
)


## Configuration — LSTM rand_C

In [ ]:
DATA_SET = 'rand_A'
NOTEBOOK_NAME = '3.0-lstm-rand-A-colab'

# ── Hyperparameters ──
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 25
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.1
BASE_LR = 1e-3
BASE_BATCH = 512

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)


## Load data

In [ ]:
df_train, df_val, df_test = load_split_bundle(CLEAN_DATA, DATA_SET)

# Load full ordered data for sequence construction
df_full = read_parquet_safe(INTERIM_DATA / '03-data-merge-feature.parquet')

print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')
print(f'Full data for sequences: {len(df_full):,} rows')

## GPU auto-detect

In [ ]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)


## Fit Hull-White benchmark (once)

In [ ]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)
hw_coef = hw['coef']


## Train across feature sets

In [ ]:
OUTPUT_DIR = ROOT / 'output' / NOTEBOOK_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_by_fs = {}
df_sorted_ref = None  # will be set on first feature set

for fs_name, feature_cols in FEATURE_SETS.items():
    print_feature_set_summary(fs_name, 0, 0, 0, feature_cols)

    # Build sequences from full ordered data, assign by target row's split
    seq_data = assign_sequences_to_splits(
        df_full, df_train, df_val, df_test,
        feature_cols=feature_cols, target=TARGET, lookback=LOOKBACK,
    )

    X_tr, y_tr = seq_data['X_train'], seq_data['y_train']
    X_va, y_va = seq_data['X_val'], seq_data['y_val']
    X_te, y_te = seq_data['X_test'], seq_data['y_test']
    test_idx = seq_data['test_indices']
    df_sorted = seq_data['df_sorted']
    if df_sorted_ref is None:
        df_sorted_ref = df_sorted

    print(f'  Sequences — Train: {len(X_tr):,}  Val: {len(X_va):,}  Test: {len(X_te):,}')

    # Scale
    X_tr_sc, X_va_sc, X_te_sc, scaler = scale_sequences(X_tr, X_va, X_te)

    # DataLoaders with adaptive batch size
    BATCH_SIZE = compute_batch_size(len(X_tr), CFG['MAX_BATCH'])
    INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5
    print_config(CFG, BATCH_SIZE, INIT_LR, len(X_tr), MAX_EPOCHS, PATIENCE, WARMUP_EPOCHS, LOOKBACK)

    train_ds = SequenceDataset(X_tr_sc, y_tr)
    val_ds = SequenceDataset(X_va_sc, y_va)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

    # Build model
    n_feat = len(feature_cols)
    model = LSTMModel(
        n_features=n_feat,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        seed=SEED,
    ).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  LSTM params: {n_params:,}')

    # Train
    train_result = train_seq_model(
        model, train_loader, val_loader,
        device=DEVICE, amp_dtype=AMP_DTYPE, use_amp=USE_AMP,
        max_epochs=MAX_EPOCHS, patience=PATIENCE,
        lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
        init_lr=INIT_LR, warmup_epochs=WARMUP_EPOCHS,
    )

    # Predict on test
    y_pred = predict_seq(train_result['model'], X_te_sc, DEVICE, AMP_DTYPE, USE_AMP)

    # HW SSE aligned to this feature set's test rows
    _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted, test_idx)

    from src.metrics import metrics, gain
    met = metrics(y_te, y_pred)
    g = gain(met['SSE'], hw_sse) * 100
    print(f'  {fs_name}: SSE={met["SSE"]:.4f}  RMSE={met["RMSE"]:.6f}  '
          f'Gain={g:+.2f}%  epochs={train_result["epochs"]}  '
          f'time={train_result["training_time"]:.1f}s')

    results_by_fs[fs_name] = {
        'model': train_result['model'],
        'y_pred': y_pred,
        'y_true': y_te,
        'test_indices': test_idx,
        'history': train_result['history'],
        'epochs': train_result['epochs'],
        'training_time': train_result['training_time'],
        'scaler': scaler,
    }

    # Free memory
    del X_tr, X_va, X_te, X_tr_sc, X_va_sc, X_te_sc
    del train_ds, val_ds, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Save results

In [ ]:
summary = save_seq_run(
    OUTPUT_DIR,
    results_by_fs=results_by_fs,
    hw_coef=hw_coef,
    df_sorted=df_sorted_ref,
)
print('\nMetrics Summary:')
summary


## Summary

In [ ]:
total_time = sum(r['training_time'] for r in results_by_fs.values())
print(f'\nLSTM on {DATA_SET} — total training time: {total_time / 60:.1f} min')
for fs_name, res in results_by_fs.items():
    from src.metrics import metrics, gain
    met = metrics(res['y_true'], res['y_pred'])
    _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted_ref, res['test_indices'])
    g = gain(met['SSE'], hw_sse) * 100
    print(f'  {fs_name}: SSE={met["SSE"]:.4f}  Gain={g:+.2f}%  epochs={res["epochs"]}')


## Cleanup

In [ ]:
del results_by_fs, df_full, df_sorted_ref
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time / 60:.1f} min')

if IN_COLAB:
    runtime.unassign()
